# Vision Transformer (ViT) — GTSRB multitask classifier

A Vision Transformer built from scratch in Keras for the same multitask problem as
`02_MobileNetV2.ipynb`: two softmax heads — `sign_class` (43 GTSRB classes) and
`sign_color` (4: red / blue / yellow / white_black, derived from `Meta.csv`).

Unlike MobileNetV2, there's no off-the-shelf ImageNet-pretrained ViT in
`tf.keras.applications`, so this notebook implements the architecture directly:
patch embedding (via `tf.image.extract_patches`), a learnable `[CLS]` token,
learnable position embeddings, and a stack of pre-norm Transformer encoder blocks
(multi-head self-attention + MLP), following Dosovitskiy et al., 2021
("An Image is Worth 16x16 Words").

Key difference from `02_MobileNetV2.ipynb`'s preprocessing bug (inconsistent
`/255` vs `preprocess_input`): here normalisation is a `Rescaling(1/255)` layer
*inside* the model, so training and inference can never disagree about it.

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score


## Config

In [ ]:
IMG_SIZE = (128, 128)      # same crop size as 02_MobileNetV2, for a fair comparison
PATCH_SIZE = 16            # 128 / 16 -> 8x8 = 64 patches per image
NUM_PATCHES = (IMG_SIZE[0] // PATCH_SIZE) * (IMG_SIZE[1] // PATCH_SIZE)
PROJECTION_DIM = 128       # transformer hidden size — kept small for CPU training
NUM_HEADS = 4
TRANSFORMER_LAYERS = 6
MLP_DIM = PROJECTION_DIM * 2
DROPOUT_RATE = 0.1
BATCH_SIZE = 32
NUM_CLASSES = 43
EPOCHS = 25

sign_names = ["limit_zone_20","limit_zone_30","limit_zone_50","limit_zone_60","limit_zone_70",
              "limit_zone_80","end_of_speed_limit","limit_zone_100","limit_zone_120",
              "no_passing","no_passing_for_trucks","right_of_way","priority_road",
              "yield_right_of_way","stop","prohibited_for_all_vehicles","tractors_and_trucks_prohibited",
              "entry_prohibited","danger","single_curve_left","single_curve_right","double_curve",
              "rough_road","slippery_road","road_narrows","construction_site","signal_lights_ahead","pedestrian_crosswalk_ahead",
              "children","bicycle_crossing","snow_ahead","wild_animal_crossing","end_of_all_restrictions",
               "mandatory_right","mandatory_left","mandatory_ahead","mandatory_ahead_right",
              "mandatory_ahead_left","mandatory_down_right","mandatory_down_left","traffic_circle","end_of_no_passing_zone",
              "end_of_no_passing_zone_trucks"]


## Data indexing

Same GTSRB CSV-driven indexing as `02_MobileNetV2.ipynb`: read `Train.csv` / `Test.csv`,
attach the colour label from `Meta.csv`, and keep the ROI box normalised for
`tf.image.crop_and_resize`. `Train.csv` is stratified 80/20 into train/validation;
`Test.csv` is the official GTSRB held-out test set, used only for final evaluation.

In [ ]:
directory = "/Users/maliya/Desktop/dissertation/Vista/data/raw/consolidated/gtsrb-german-traffic-sign"

meta = pd.read_csv(os.path.join(directory, "Meta.csv"))
COLOR_NAMES = ["red", "blue", "yellow", "white_black"]
NUM_COLORS = len(COLOR_NAMES)
class_to_color = meta.set_index("ClassId")["ColorId"].astype(int).to_dict()


def index_csv(csv_name):
    df = pd.read_csv(os.path.join(directory, csv_name))
    df["abspath"] = df["Path"].apply(lambda p: os.path.join(directory, p))
    df["color"] = df["ClassId"].map(class_to_color).astype(int)
    df["box_y1"] = df["Roi.Y1"] / df["Height"]
    df["box_x1"] = df["Roi.X1"] / df["Width"]
    df["box_y2"] = df["Roi.Y2"] / df["Height"]
    df["box_x2"] = df["Roi.X2"] / df["Width"]
    return df


train_df = index_csv("Train.csv")
test_df = index_csv("Test.csv")

train_rows, valid_rows = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df["ClassId"]
)
print(f"{len(train_rows)} train / {len(valid_rows)} val / {len(test_df)} test images, "
      f"{train_df['ClassId'].nunique()} classes, {NUM_COLORS} colours")


## `tf.data` pipeline

Decode + ROI-crop once per image, cache the compact uint8 crop to disk, then batch.
Pixel normalisation to `[0, 1]` is *not* done here — it's a `Rescaling` layer inside
the model (see below), so it's guaranteed identical between training and inference.

In [ ]:
import tempfile

AUTOTUNE = tf.data.AUTOTUNE
_cache_root = tempfile.mkdtemp(prefix="gtsrb_vit_cache_")


def make_dataset(rows, *, training, cache_name):
    paths = rows["abspath"].to_numpy()
    boxes = rows[["box_y1", "box_x1", "box_y2", "box_x2"]].to_numpy("float32")
    y_cls = tf.keras.utils.to_categorical(rows["ClassId"].to_numpy(), NUM_CLASSES)
    y_col = tf.keras.utils.to_categorical(rows["color"].to_numpy(), NUM_COLORS)

    ds = tf.data.Dataset.from_tensor_slices((paths, boxes, y_cls, y_col))

    def decode(path, box, yc, yk):
        img = tf.io.decode_png(tf.io.read_file(path), channels=3)
        img = tf.image.crop_and_resize(img[tf.newaxis], box[tf.newaxis], [0], IMG_SIZE)[0]
        img = tf.cast(tf.round(img), tf.uint8)          # cache compact uint8 crops
        return img, yc, yk

    ds = ds.map(decode, num_parallel_calls=AUTOTUNE).cache(
        os.path.join(_cache_root, cache_name)
    )
    if training:
        ds = ds.shuffle(4096, reshuffle_each_iteration=True)

    def prep(img, yc, yk):
        img = tf.cast(img, tf.float32)   # model's Rescaling(1/255) layer normalises
        return img, {"sign_class": yc, "sign_color": yk}

    ds = ds.map(prep, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_dataset = make_dataset(train_rows, training=True, cache_name="train")
valid_dataset = make_dataset(valid_rows, training=False, cache_name="valid")
test_dataset = make_dataset(test_df, training=False, cache_name="test")


## Vision Transformer building blocks

- `Patches`: splits an image into non-overlapping `PATCH_SIZE x PATCH_SIZE` patches
  and flattens each into a vector, via `tf.image.extract_patches`.
- `PatchEncoder`: linearly projects each patch to `PROJECTION_DIM`, prepends a
  learnable `[CLS]` token, and adds learnable position embeddings.
- `transformer_encoder_block`: a pre-norm Transformer block — LayerNorm ->
  multi-head self-attention -> residual -> LayerNorm -> GELU MLP -> residual.

Both layers are registered with `register_keras_serializable` so the model can be
saved/loaded with the native Keras format.

In [ ]:
@keras.saving.register_keras_serializable(package="vit")
class Patches(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        return tf.reshape(patches, [batch_size, -1, patch_dims])

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config


@keras.saving.register_keras_serializable(package="vit")
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches + 1, output_dim=projection_dim
        )

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            shape=(1, 1, self.projection_dim),
            initializer="random_normal",
            trainable=True,
            name="cls_token",
        )
        super().build(input_shape)

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        positions = tf.range(start=0, limit=self.num_patches + 1, delta=1)
        projected = self.projection(patch)                       # (B, N, D)
        cls_tokens = tf.repeat(self.cls_token, batch_size, axis=0)
        projected = tf.concat([cls_tokens, projected], axis=1)   # (B, N+1, D)
        return projected + self.position_embedding(positions)

    def get_config(self):
        config = super().get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config


@keras.saving.register_keras_serializable(package="vit")
class ClsTokenExtractor(layers.Layer):
    """Pulls the [CLS] token (position 0) out of the encoded patch sequence."""

    def call(self, encoded_patches):
        return encoded_patches[:, 0]


def transformer_encoder_block(x, *, projection_dim, num_heads, mlp_dim, dropout_rate):
    x1 = layers.LayerNormalization(epsilon=1e-6)(x)
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout_rate
    )(x1, x1)
    x2 = layers.Add()([attn_output, x])

    x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
    x3 = layers.Dense(mlp_dim, activation=tf.nn.gelu)(x3)
    x3 = layers.Dropout(dropout_rate)(x3)
    x3 = layers.Dense(projection_dim)(x3)
    x3 = layers.Dropout(dropout_rate)(x3)
    return layers.Add()([x3, x2])


## Multitask ViT model

In [ ]:
def build_vit_classifier():
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inputs)   # pixels -> [0, 1]

    patches = Patches(PATCH_SIZE)(x)
    encoded_patches = PatchEncoder(NUM_PATCHES, PROJECTION_DIM)(patches)

    for _ in range(TRANSFORMER_LAYERS):
        encoded_patches = transformer_encoder_block(
            encoded_patches,
            projection_dim=PROJECTION_DIM,
            num_heads=NUM_HEADS,
            mlp_dim=MLP_DIM,
            dropout_rate=DROPOUT_RATE,
        )

    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    cls_representation = ClsTokenExtractor()(representation)
    cls_representation = layers.Dropout(0.3)(cls_representation)

    features = layers.Dense(128, activation=tf.nn.gelu)(cls_representation)
    features = layers.Dropout(0.3)(features)

    sign_class_output = layers.Dense(NUM_CLASSES, activation="softmax", name="sign_class")(features)
    sign_color_output = layers.Dense(NUM_COLORS, activation="softmax", name="sign_color")(features)

    return models.Model(inputs, [sign_class_output, sign_color_output], name="vit_multitask")


model = build_vit_classifier()
model.summary()


## Train

ViT has no ImageNet pretraining here (unlike MobileNetV2), so it's trained from
scratch: `AdamW` with weight decay plus a linear warmup / cosine decay learning-rate
schedule, which is standard practice for training Transformers from scratch and
stabilises the early epochs.

In [ ]:
steps_per_epoch = len(train_rows) // BATCH_SIZE

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0,
    decay_steps=steps_per_epoch * EPOCHS,
    warmup_target=1e-3,
    warmup_steps=steps_per_epoch * 3,
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4),
    loss={"sign_class": "categorical_crossentropy", "sign_color": "categorical_crossentropy"},
    loss_weights={"sign_class": 1.0, "sign_color": 0.3},
    metrics={"sign_class": "accuracy", "sign_color": "accuracy"},
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_sign_class_accuracy", mode="max",
                                      patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint("vit_multitask_best.keras", monitor="val_sign_class_accuracy",
                                        mode="max", save_best_only=True, verbose=1),
]

history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.save("vit_multitask.keras")


## Evaluate

Validation metrics from Keras, plus precision/recall/F1 (weighted average) on the
official GTSRB **test** set for both heads — comparable to the CNN notebook's
`calculate_results` and to `02_MobileNetV2.ipynb`.

In [ ]:
val_metrics = model.evaluate(valid_dataset, return_dict=True, verbose=0)
print("Validation:", {k: round(v, 4) for k, v in val_metrics.items()})

test_metrics = model.evaluate(test_dataset, return_dict=True, verbose=0)
print("Test:", {k: round(v, 4) for k, v in test_metrics.items()})


In [ ]:
def calculate_results(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred) * 100
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


class_probs, color_probs = model.predict(test_dataset, verbose=0)
y_class_true = test_df["ClassId"].to_numpy()
y_color_true = test_df["color"].to_numpy()
y_class_pred = np.argmax(class_probs, axis=1)
y_color_pred = np.argmax(color_probs, axis=1)

print("sign_class:", {k: round(v, 4) for k, v in calculate_results(y_class_true, y_class_pred).items()})
print("sign_color:", {k: round(v, 4) for k, v in calculate_results(y_color_true, y_color_pred).items()})


## Single-image inference helper

Mirrors `recognize_feature` from `02_MobileNetV2.ipynb` — normalisation lives in
the model (`Rescaling`), so callers just pass raw `[0, 255]` pixels.

In [ ]:
def recognize_feature(image_path, model):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_batch = np.expand_dims(img_array, axis=0)

    class_probs, color_probs = model.predict(img_batch, verbose=0)
    class_index = np.argmax(class_probs[0])
    confidence = class_probs[0][class_index]
    color_index = np.argmax(color_probs[0])
    color_confidence = color_probs[0][color_index]

    return class_index, confidence, COLOR_NAMES[color_index], color_confidence


sample_path = os.path.join(directory, test_df.iloc[0]["Path"])
class_index, confidence, color_name, color_confidence = recognize_feature(sample_path, model)
print(sign_names[class_index], f"{confidence:.3f}", color_name, f"{color_confidence:.3f}",
      "| true:", sign_names[test_df.iloc[0]['ClassId']])
